In [8]:
import subsettools as st
import hf_hydrodata as hf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from parflow.tools.io import read_pfb
from parflow.tools.io import read_pfb,write_pfb, read_clm
import glob
from collections import defaultdict
from pathlib import Path
import re

# hf.register_api_pin("email", "hf pin")

# subset forcing

In [7]:
ij_huc_bounds_ET, mask_ET = st.define_huc_domain(hucs=["14020001"], grid="conus2") 
print(f"bounding box: {ij_huc_bounds_ET}")
nj = ij_huc_bounds_ET[3] - ij_huc_bounds_ET[1]
ni = ij_huc_bounds_ET[2] - ij_huc_bounds_ET[0]
print(f"nj: {nj}")
print(f"ni: {ni}")

bounding box: (1360, 1568, 1425, 1618)
nj: 50
ni: 65


In [10]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

filepaths = st.subset_forcing(
    ij_huc_bounds_ET,
    grid="conus2",
    start="1990-10-01",
    end="2025-10-01",
    dataset="CW3E",
    write_dir="~/EastTaylor_inputs/precip_temp_90_25",
    forcing_vars = ('precipitation', 'air_temp')
)


Reading precipitation pfb sequence
Reading air_temp pfb sequence
Finished writing precipitation to folder
Finished writing air_temp to folder


# calculate averages

In [36]:
forcing_dir = Path(
    "~/EastTaylor_inputs/precip_temp_90_25"
)

# change these for a different averaging window
start_water_year = 2000
end_water_year = 2020   # inclusive

forcing_start = pd.Timestamp("1990-10-01 00:00")


def get_start_hour(filepath):

    match = re.search(r"\.(\d+)_to_(\d+)\.pfb$", filepath.name)

    return int(match.group(1))


def get_file_start_time(filepath):
    """
    000001_to_000024 starts at forcing_start.
    """
    start_hour = get_start_hour(filepath)
    return forcing_start + pd.Timedelta(hours=start_hour - 1)


def get_water_year(timestamp):
    if timestamp.month >= 10:
        return timestamp.year + 1
    else:
        return timestamp.year

In [38]:
precip_files = sorted(forcing_dir.glob("CW3E.APCP.*.pfb"))

annual_precip_sums = defaultdict(float)

for f in precip_files:
    file_start_time = get_file_start_time(f)
    wy = get_water_year(file_start_time)

    if wy < start_water_year or wy > end_water_year:
        continue

    data = read_pfb(str(f))  # (24, ny, nx)

    # gives mm/year.
    annual_precip_sums[wy] += (data.mean(axis=(1, 2)) * 3600).sum() # converting from s

avg_annual_precip_sum = np.mean([
    annual_precip_sums[wy]
    for wy in range(start_water_year, end_water_year + 1)
])

print(f"Average annual precipitation sum, WY{start_water_year}-WY{end_water_year}:")
print(avg_annual_precip_sum)

Average annual precipitation sum, WY2000-WY2020:
630.7299961597378


In [39]:
temp_files = sorted(forcing_dir.glob("CW3E.Temp.*.pfb"))

annual_temp_sums = defaultdict(float)
annual_temp_counts = defaultdict(int)

for f in temp_files:
    file_start_time = get_file_start_time(f)
    wy = get_water_year(file_start_time)

    if wy < start_water_year or wy > end_water_year:
        continue

    data = read_pfb(str(f))

    annual_temp_sums[wy] += data.sum()
    annual_temp_counts[wy] += data.size

annual_temp_means = {
    wy: annual_temp_sums[wy] / annual_temp_counts[wy]
    for wy in range(start_water_year, end_water_year + 1)
}

avg_annual_temp_mean = np.mean(list(annual_temp_means.values()))

print(f"Average annual temperature mean, WY{start_water_year}-WY{end_water_year}:")
print(avg_annual_temp_mean)

print("Celsius is:")
print(avg_annual_temp_mean - 273.15)

Average annual temperature mean, WY2000-WY2020:
274.78494109147
Celsius is:
1.6349410914700115
